# Reconnaissance d'expressions faciales — FER2013

**Projet Deep Learning** — Étudiant A / Étudiant B

**Objectif :** entraîner et comprendre un modèle qui reconnaît 7 expressions faciales
(angry, disgust, fear, happy, neutral, sad, surprise) à partir d'images de visages 48×48 en niveaux de gris.

**Dataset :** FER2013 — Kaggle `msambare/fer2013`

**Plan du notebook** (construit progressivement, une cellule par micro-étape) :
1. Exploration et préparation du dataset
2. Baseline dense
3. CNN
4. Entraînement
5. Évaluation et analyse des erreurs
6. Expériences (≥ 3)


### Étape 0 — Configuration commune et téléchargement des données

Cette cellule est lancée **en premier** ; toutes les suivantes en dépendent.

1. **Racine du projet.** En local, c'est le dossier qui contient `.gitignore` : le notebook marche qu'on le lance depuis la racine ou depuis `notebooks/`. Sur Colab, si le dépôt n'a pas été cloné, il n'y a pas de `.gitignore` : on reste alors dans le dossier courant (`/content`).
2. **Constantes partagées.** `DATA_DIR`, `CLASS_NAMES`, `IMAGE_SIZE` et `SEED` sont définies **une seule fois ici** et réutilisées partout : changer une valeur à un endroit la change pour tout le notebook.
3. **Graine aléatoire (*seed*).** `keras.utils.set_random_seed(42)` fixe le hasard de Python, NumPy et TensorFlow (découpe train/validation, poids initiaux, mélange des lots). Deux expériences ne diffèrent alors que par ce qu'on a volontairement changé — indispensable pour comparer nos ≥ 3 expériences. (Sur GPU, quelques opérations restent légèrement non déterministes : de petites variations restent possibles.)
4. **Téléchargement idempotent.** `data/` n'est pas dans Git (≈ 200 Mo). Si `data/fer2013/train` et `data/fer2013/test` existent déjà, la cellule ne télécharge rien ; sinon elle récupère FER2013 avec `kagglehub` (bibliothèque officielle de Kaggle). La relancer 10 fois donne le même résultat qu'une seule fois.

In [ ]:
# Étape 0 : configuration commune + téléchargement de FER2013 s'il manque
import os
import shutil
from pathlib import Path

from tensorflow import keras

# --- 1) Racine du projet ---
# On remonte les dossiers jusqu'à trouver .gitignore (racine du dépôt).
racine = Path.cwd()
while racine != racine.parent and not (racine / ".gitignore").exists():
    racine = racine.parent
if not (racine / ".gitignore").exists():
    # Pas de dépôt (ex. notebook ouvert seul sur Colab) : on reste dans le dossier courant.
    racine = Path.cwd()
os.chdir(racine)  # les cellules suivantes utilisent des chemins relatifs à cette racine
print("Racine du projet :", racine)

# --- 2) Constantes partagées par tout le notebook ---
DATA_DIR = Path("data/fer2013")
# Cet ordre fixe la position du 1 dans le vecteur one-hot :
# 0 = angry, 1 = disgust, 2 = fear, 3 = happy, 4 = neutral, 5 = sad, 6 = surprise.
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
IMAGE_SIZE = (48, 48)  # taille des images FER2013 (vérifiée à l'étape 1b)
SEED = 42

# --- 3) Graine aléatoire : rend les résultats reproductibles ---
keras.utils.set_random_seed(SEED)

# --- 4) Téléchargement, seulement si les données manquent ---
if (DATA_DIR / "train").is_dir() and (DATA_DIR / "test").is_dir():
    print(f"Données déjà présentes dans {DATA_DIR} — aucun téléchargement.")
else:
    import kagglehub  # déjà installé sur Colab ; en local : pip install kagglehub

    print("Téléchargement du dataset msambare/fer2013 via kagglehub…")
    source = Path(kagglehub.dataset_download("msambare/fer2013"))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "test"]:
        if not (DATA_DIR / split).exists():  # jamais réécrit si déjà copié
            shutil.copytree(source / split, DATA_DIR / split)
    print("Dataset copié dans", DATA_DIR.resolve())

Avant de charger les données, on inspecte l'arborescence de `data/fer2013` : chaque dossier de classe sert d'étiquette (label), c'est cette structure que Keras lira.

In [ ]:
# Micro-étape 1a : compter les images de chaque classe
# Chaque dossier de classe (data/fer2013/train/happy, ...) contient les images de cette classe :
# le nom du dossier EST l'étiquette.
for split in ["train", "test"]:
    total = 0
    for classe in CLASS_NAMES:
        dossier = DATA_DIR / split / classe
        n = len(list(dossier.glob("*.jpg")))  # nombre de fichiers .jpg dans le dossier
        total = total + n
        print(f"{split}/{classe} : {n}")
    print(f"{split} total : {total}")
    print()

### Étape 1b — Regarder les images et vérifier le format

**Pourquoi ?** Un modèle apprend à partir de pixels : s'il reçoit des images de tailles ou de modes différents, la couche `Flatten` produira des vecteurs de longueurs différentes et l'entraînement échouera (ou buggera silencieusement).

On fait donc deux choses :
1. **On affiche une grille de 7 classes × 5 images** tirées de `data/fer2013/train/<classe>/` : on vérifie *à l'œil* que les dossiers contiennent bien des visages et que les classes sont cohérentes (utile aussi pour l'oral : montrer des exemples réels).
2. **On vérifie le format sur *toutes* les images** (train + test) avec PIL : taille `(48, 48)`, mode `L` (niveaux de gris) et extension `.jpg`. Un `Counter` (bibliothèque standard Python) compte combien de fois chaque valeur apparaît ; on liste les exceptions plutôt que de les cacher.

In [ ]:
# Micro-étape 1b : exemples d'images + vérification du format (taille, mode, extension)
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

N_EXEMPLES = 5  # nombre d'images affichées par classe

# --- 1) grille 7 lignes (classes) x 5 colonnes (exemples) -------------------
fig, axes = plt.subplots(len(CLASS_NAMES), N_EXEMPLES, figsize=(10, 14))
for ligne, classe in enumerate(CLASS_NAMES):
    # on trie les fichiers pour que la grille soit la même à chaque exécution
    fichiers = sorted((DATA_DIR / "train" / classe).glob("*.jpg"))[:N_EXEMPLES]
    for col, chemin in enumerate(fichiers):
        with Image.open(chemin) as img:  # le `with` ferme l'image automatiquement
            axes[ligne, col].imshow(img, cmap="gray")  # images en niveaux de gris
        axes[ligne, col].set_xticks([])  # on masque les graduations : on veut voir l'image
        axes[ligne, col].set_yticks([])
    # nom de la classe écrit à gauche de la ligne
    axes[ligne, 0].set_ylabel(classe, rotation=0, ha="right", va="center")

fig.suptitle("FER2013 — 5 exemples par classe (split train)", fontsize=14)
plt.tight_layout()
plt.show()

# --- 2) vérification du format sur TOUTES les images (train + test) --------
# Counter compte les valeurs rencontrées, ex. {(48, 48): 35887}
tailles, modes, extensions = Counter(), Counter(), Counter()
anomalies = []  # (chemin, description du problème)
n_images = 0

for split in ["train", "test"]:
    for classe in CLASS_NAMES:
        dossier = DATA_DIR / split / classe
        for chemin in sorted(dossier.iterdir()):  # tous les fichiers du dossier de classe
            n_images = n_images + 1
            extension = chemin.suffix.lower()  # ".jpg", ".png", ...
            try:
                with Image.open(chemin) as img:
                    taille = img.size
                    mode = img.mode
            except OSError:  # fichier corrompu ou illisible : on le note et on continue
                anomalies.append((chemin, "illisible"))
                continue
            tailles[taille] += 1
            modes[mode] += 1
            extensions[extension] += 1
            if taille != IMAGE_SIZE or mode != "L" or extension != ".jpg":
                anomalies.append((chemin, f"taille {taille}, mode {mode}, extension {extension}"))

print(f"Images vérifiées : {n_images} (train + test)")
print(f"Tailles observées    : {dict(tailles)}")
print(f"Modes PIL observés   : {dict(modes)}")
print(f"Extensions observées : {dict(extensions)}")
if anomalies:
    print(f"\n{len(anomalies)} exception(s) trouvée(s) sur {n_images} images :")
    for chemin, probleme in anomalies[:20]:  # on n'affiche que les 20 premières
        print(f"  - {chemin} -> {probleme}")
else:
    print(f"\nFormat : 48×48, mode L, .jpg — vérifié sur {n_images} images, aucune exception.")

### Étape 1c — Construire les datasets d'entraînement, de validation et de test

**Pourquoi ?** Jusqu'ici on a seulement *regardé* les fichiers. Le modèle a besoin d'un `tf.data.Dataset` : un flux de lots d'images + d'étiquettes, prêt pour `model.fit()`.

Cinq points à comprendre :

1. **Normalisation** — les pixels sont des entiers entre 0 et 255. On les divise par 255 pour les ramener dans `[0, 1]` : des entrées petites et à la même échelle rendent l'entraînement plus stable et, en général, plus rapide.
2. **Encodage one-hot** — `label_mode="categorical"` transforme l'étiquette `"happy"` en vecteur `[0, 0, 0, 1, 0, 0, 0]` de 7 valeurs (la position du 1 suit l'ordre de `CLASS_NAMES`). C'est la forme attendue par une sortie `softmax` avec la perte `categorical_crossentropy`.
3. **Validation** — 20 % de `data/fer2013/train` est mis de côté (5 741 images) pour mesurer la généralisation pendant l'entraînement et repérer le surapprentissage. Ces images n'entraînent jamais les poids : elles servent seulement à décider quand s'arrêter (`early stopping`, choix du meilleur modèle).
4. **Test** — `data/fer2013/test` (7 178 images) reste intact, sans mélange ni découpe : il est réservé à l'évaluation finale, une seule fois, à la toute fin du projet. Si on l'utilisait pendant le développement, on « tricherait » sans le savoir.

Rappel de l'étape 1b : toutes les images font déjà 48×48 en niveaux de gris, donc la forme d'une image est `(48, 48, 1)` et aucune redimensionnement n'est nécessaire.

5. **Performance** — `.cache()` garde les images en mémoire après la première époque (on ne relit plus les 28 000 fichiers JPEG à chaque époque) et `.prefetch()` prépare le lot suivant pendant que le modèle calcule sur le lot courant. Ça ne change pas les résultats, seulement la vitesse.

In [ ]:
# Micro-étape 1c : construire les datasets TensorFlow (train / validation / test)
import tensorflow as tf

BATCH_SIZE = 64         # nombre d'images par lot
VALIDATION_SPLIT = 0.2  # 20 % du dossier train est réservé à la validation

# --- 1) train + validation : une seule passe sur data/fer2013/train ---
# subset="both" renvoie directement un tuple (train_ds, val_ds).
# seed=SEED rend la découpe reproductible : la même image tombe toujours du même côté.
# color_mode="grayscale"  -> 1 seul canal, donc une forme (48, 48, 1).
# label_mode="categorical" -> étiquette encodée en one-hot (vecteur de 7 valeurs).
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    seed=SEED,
)

# --- 2) test : data/fer2013/test, utilisé tel quel, sans découpe ---
# shuffle=False garde l'ordre des fichiers, pour une évaluation stable et reproductible.
test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# --- 3) nombre d'images réellement présent dans chaque ensemble ---
# .file_paths n'existe que sur le dataset d'origine : on le lit avant le .map ci-dessous.
print("Images en train      :", len(train_ds.file_paths))  # attendu : 22968
print("Images en validation :", len(val_ds.file_paths))    # attendu : 5741
print("Images en test       :", len(test_ds.file_paths))   # attendu : 7178

# --- 4) normalisation : les pixels passent de l'échelle [0, 255] à [0, 1] ---
def normaliser(images, labels):
    return images / 255.0, labels

# .map applique la fonction à chaque lot ; .cache + .prefetch accélèrent la lecture.
train_ds = train_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)

# --- 5) contrôle : forme d'un lot et valeurs des pixels ---
images_batch, labels_batch = next(iter(train_ds))  # premier lot : 64 images + étiquettes
print("Forme du lot d'images  :", images_batch.shape)  # attendu : (64, 48, 48, 1)
print("Forme du lot de labels :", labels_batch.shape)  # attendu : (64, 7)
print("Valeur min des pixels  :", float(images_batch.numpy().min()))  # entre 0 et 1
print("Valeur max des pixels  :", float(images_batch.numpy().max()))  # entre 0 et 1